In [ ]:
import fsspec
import icechunk
import numpy as np
import obstore
import xarray
from obspec_utils.registry import ObjectStoreRegistry
from virtual_tiff import VirtualTIFF
from virtualizarr import open_virtual_dataset

# Building the AHN DSM virtual Zarr store

## Introduction

This notebook shows how to build a virtual Zarr store for the high-resolution (7.5 cm) Digital Surface Model (DSM) derived from the Actueel Hoogtebestand Nederland (AHN), version 4. As data source, we use Cloud-Optimized GeoTIFF (COG) tiles derived from the original 1x1 km GeoTIFF tiles and stored on a public bucket on SURF object storage.

The virtual Zarr store is made available via the SURF object store, in three different formats:
* a JSON kerchunk reference file;
* a Parquet kerchunk reference file;
* an Icechunk repository.

## Parsing the references

Let's get the list of all COGs (NOTE: credentials need to be set via environment variables, see [README.md](../README.md)):

In [ ]:
bucket = "s3://cloud-nes-benchmarks"
fs = fsspec.filesystem("s3")
paths = fs.glob(f"{bucket}/ahn/4/DSM05/COG-400-LZW/*.TIF")

We parse references to the data chunks in the COGs using VirtualiZarr and virtual-tiff:

In [ ]:
store = obstore.store.from_url(bucket)
registry = ObjectStoreRegistry({bucket: store})
# only extract chunk references for the first ifd (the data at highest resolution).
# use instead `ifd_layout="nested"` and `open_virtual_datatree` to extract refs for all overviews.
parser = VirtualTIFF(ifd=0) 

virtual-tiff currently does not parse coordinates from the TIFF files, so these need to be build "manually" from the attributes in order to be able to concatenate the manifest arrays:

In [ ]:
def _generate_coords(attrs, shape):
    """Produce coordinate arrays for given variable

    Taken and adapted from kerchunk: https://github.com/fsspec/kerchunk/blob/3cd5ddc00f22bf9ec8664552aa49f3e992c486fb/kerchunk/tiff.py#L110
    
    Parameters
    ----------
    attrs: dict
        Containing the geoTIFF tags, probably the root group of the dataset
    shape: tuple[int]
        The array size in numpy (C) order
    """
    height, width = shape[-2:]
    xscale, yscale, zscale = attrs["model_pixel_scale"][:3]
    x0, y0, z0 = attrs["model_tiepoint"][3:6]
    out = {}
    out["x"] = np.arange(width) * xscale + x0 + xscale / 2
    out["y"] = np.arange(height) * -yscale + y0 - yscale / 2
    if len(shape) > 2:
        out["z"] = np.arange(shape[-3]) * zscale + z0 + zscale / 2
    return out

def _open_virtual_dataset(path, registry, parser):
    ds = open_virtual_dataset(path, registry=registry, parser=parser)
    coords = _generate_coords(ds["0"].attrs, ds["0"].shape)
    return ds.assign_coords(coords)

Extract all references:

In [ ]:
vds = [
    _open_virtual_dataset(f"s3://{path}", registry, parser)
    for path in paths
]

## Combine references as a "virtual" data cube

We can now use Xarray's functionality to merge all manifest arrays as a single (virtual) data cube:

In [ ]:
vds_combined = xarray.combine_by_coords(vds, join="outer", combine_attrs="drop_conflicts")
vds_combined

Let's use the public HTTP links within the references, so that the reference files can be opened without S3 settings:

In [ ]:
def rename_path(src: str) -> str:
    return src.replace("s3://", "https://objectstore.surf.nl/")
    
vds_renamed = vds_combined.vz.rename_paths(rename_path)

## Persisting the virtual Zarr store

The virtual Zarr store is then saved as:

1. A JSON kerchunk reference file. Note that the VirtualiZarr implementation currently only supports a local filesystem for the output, so we save the JSON file locally and upload it to the object store using AWS CLI: 

In [ ]:
vds_renamed.vz.to_kerchunk("refs-COG-400-LZW.json", format="json") # to be uploaded manually

In [ ]:
!aws s3 cp refs-COG-400-LZW.json s3://cloud-nes-benchmarks/ahn/4/DSM05/refs-COG-400-LZW.json

2. A Parquet kerchunk reference file (S3 output is supported):

In [ ]:
vds_renamed.vz.to_kerchunk("s3://cloud-nes-benchmarks/ahn/4/DSM05/refs-COG-400-LZW.parquet", format="parquet")

3. An Icechunk repository. We initialize the repository:

In [ ]:
storage = icechunk.s3_storage(
    bucket="cloud-nes-benchmarks", 
    prefix="ahn/4/DSM05/COG-400-LZW-icechunk",    
    from_env=True,
)

config = icechunk.RepositoryConfig.default()
config.set_virtual_chunk_container(
    icechunk.VirtualChunkContainer(
        url_prefix="https://objectstore.surf.nl/cloud-nes-benchmarks/",
        store=icechunk.http_store(),
        name="http",
    )
)

repo = icechunk.Repository.create(
    storage=storage,
    config=config,
    authorize_virtual_chunk_access={
        "https://objectstore.surf.nl/cloud-nes-benchmarks/": icechunk.Credentials.HttpAccess()
    },
)
# persist the configuration, so it's available when opening the repository again
repo.save_config()

then create a new session, write references and commit changes to the repo.

In [ ]:
session = repo.writable_session("main")
vds_renamed.vz.to_icechunk(session.store)
session.commit("data for COG-400-LZW")